# Facial Expression Recognition (FER-2013)
## Regularization Experiments: Dropout, Batch Normalization & Data Augmentation

This notebook conducts systematic ablation and regularization experiments to evaluate how:
1. **Dropout**: Reduces co-adaptation of neurons and curbs overfitting.
2. **Batch Normalization (BatchNorm)**: Stabilizes internal covariate shift, accelerates convergence, and smooths optimization.
3. **Data Augmentation**: Enhances training set diversity via random horizontal flips, subtle rotations, and zooms to boost generalization.
4. **Combined Regularization**: Synergy of BatchNorm + Dropout + Data Augmentation.

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import StandardCNN, DropoutCNN, BatchNormCNN, FullRegularizedCNN
from src.data import get_torch_dataloaders, ClassDistributionAnalyzer
from src.training import RegularizationExperimentSuite

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute Device: {device}')

### 1. DataLoaders & Class Weights Setup
Prepare DataLoaders with and without data augmentation, and calculate balanced class weights.

In [ ]:
# Analyze class distribution
analyzer = ClassDistributionAnalyzer()
stats = analyzer.analyze()
print(f"Total Train Images: {stats['total_train']} | Test Images: {stats['total_test']}")
print(f"Imbalance Ratio: {stats['imbalance_ratio']}x")

# Non-Augmented DataLoaders (Baseline)
standard_loaders = get_torch_dataloaders(batch_size=64, augment_train=False)
# Augmented DataLoaders
augmented_loaders = get_torch_dataloaders(batch_size=64, augment_train=True)

print("DataLoaders successfully prepared!")

### 2. Inspecting the Models for Comparison
Notice that each model shares identical convolutional filter topologies (32 -> 64 -> 128) and classifier structure, isolating the effect of Dropout and Batch Normalization.

In [ ]:
m_standard = StandardCNN()
m_dropout = DropoutCNN()
m_bn = BatchNormCNN()
m_full = FullRegularizedCNN()

print(f"Standard CNN (No Reg) Params : {m_standard.get_num_params()[0]:,}")
print(f"Dropout CNN Params           : {m_dropout.get_num_params()[0]:,}")
print(f"BatchNorm CNN Params         : {m_bn.get_num_params()[0]:,}")
print(f"Full Regularized CNN Params  : {m_full.get_num_params()[0]:,}")

### 3. Run Regularization Experiment Suite
We can run all 5 experiments or individual experiments using `RegularizationExperimentSuite`.

In [ ]:
# Initialize Experiment Suite
suite = RegularizationExperimentSuite(batch_size=64, lr=0.001)

# Run all experiments (Set epochs=10 for full training run)
results = suite.run_all(epochs=10, patience=4)

### 4. Results & Overfitting Analysis

In [ ]:
# Display Comparative Summary Table
summary_df = suite.get_summary_dataframe()
summary_df